# Fine-tune ViT on HaGRID Hand Gesture Dataset

This notebook demonstrates how to fine-tune a Vision Transformer (ViT-B/16 or alternatives) on a subset of the HaGRID dataset for real-time hand gesture recognition.

## Prerequisites

1. **Download HaGRID Subset**: Run the download script to create a manageable subset (~2-3GB) of the dataset:
   ```bash
   python scripts/download_hagrid_subset.py \
       --save_path ./data/hagrid_subset \
       --gestures call dislike fist like ok palm peace point rock no_gesture \
       --samples_per_class 800 \
       --download_annotations
   ```

2. **Conda Environment**: Create and activate the environment with PyTorch and dependencies (see next cell)

3. **Hardware**: GPU/MPS recommended for faster training, but CPU will work for small experiments

## Dataset Info
- **Selected Gestures**: 10 classes (call, dislike, fist, like, ok, palm, peace, point, rock, no_gesture)
- **Images per class**: ~800 (distributed as 76% train, 9% val, 15% test)
- **Total size**: ~2.5 GB
- **Image format**: JPG, FullHD downsampled
- **Annotations**: CSV file with columns (image_path, label, split)

## Conda environment setup (recommended)

Run these commands in your terminal (adjust channels for your platform). This creates a conda env and installs core packages used in this notebook.

```bash
# Create environment
conda env create -f environment.yml
conda activate gesture

# Or manual setup
conda create -n gesture python=3.10 -y
conda activate gesture
conda install -c pytorch pytorch torchvision -y
conda install -c conda-forge timm pillow matplotlib seaborn pandas tqdm scikit-learn jupyter -y
```

If you prefer `pip`-only inside the environment, replace the `conda install` steps accordingly. Ensure you pick the correct PyTorch build for CUDA/MPS if you have GPU/MPS available.

In [2]:
# Quick runtime check: imports and device
import sys
import torch
import torchvision
import timm
from PIL import Image
print('Python', sys.version.split()[0])
print('PyTorch', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_built() and torch.backends.mps.is_available() else 'cpu')
print('Device:', device)
# show a couple library versions
import timm
print('timm', timm.__version__)


Python 3.10.19
PyTorch 2.5.1
Device: mps
timm 1.0.22


## Dataset: pointing `dataset_path` to your HaGrid data

This loader supports either: 1) a folder-per-class structure (`dataset_path/train/<class>/*.jpg`), or 2) a CSV file (`image_path,label`) describing the dataset. Set `DATASET_PATH` below to the root of your dataset or to a CSV file path. If your HaGrid data contains raw frames and bounding-box annotations, first run the hand-detection cropper to export per-hand images into a folder structure (recommended).

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# User configuration: update this path to your HaGrid dataset root or CSV
DATASET_PATH = './data/hagrid_subset'  # <- change me
IMG_SIZE = 224  # ViT-B/16 default input size (224x224). Adjust if using other models.
BATCH_SIZE = 32
NUM_WORKERS = 4
SEED = 42

# Generic dataset loader supporting folder-per-class or CSV with columns (image,label)
class HandGestureDataset(Dataset):
    def __init__(self, root_or_csv, transform=None):
        self.transform = transform
        if os.path.isfile(root_or_csv) and root_or_csv.endswith('.csv'):
            df = pd.read_csv(root_or_csv)
            assert {'image','label'}.issubset(set(df.columns)), 'CSV must contain image,label columns'
            self.items = [(row['image'], row['label']) for _, row in df.iterrows()]
        else:
            root = Path(root_or_csv)
            # assume folder-per-class
            classes = [p.name for p in root.iterdir() if p.is_dir()]
            classes.sort()
            self.class_to_idx = {c:i for i,c in enumerate(classes)}
            items = []
            for c in classes:
                p = root / c
                for img in p.glob('*'):
                    if img.suffix.lower() in ['.jpg','.jpeg','.png']:
                        items.append((str(img), self.class_to_idx[c]))
            self.items = items
        
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(label)

# simple transforms (replace / extend with albumentations if desired)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Visualize Sample Images

Let's visualize a few samples from each class to verify the dataset loaded correctly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if os.path.exists(ANNOTATIONS_CSV):
    # Visualize 16 random samples
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    axes = axes.flatten()
    
    # Get random samples
    samples_per_class = 2
    plot_idx = 0
    
    for cls in train_dataset.classes[:8]:  # Show up to 8 classes
        class_df = train_df[train_df['label'] == cls].sample(n=min(samples_per_class, len(train_df[train_df['label'] == cls])))
        
        for _, row in class_df.iterrows():
            if plot_idx >= 16:
                break
            
            img = Image.open(row['image_path']).convert('RGB')
            axes[plot_idx].imshow(img)
            axes[plot_idx].set_title(cls, fontsize=10)
            axes[plot_idx].axis('off')
            plot_idx += 1
    
    # Hide remaining subplots
    for idx in range(plot_idx, 16):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Dataset not loaded. Run the download script first.')

In [ ]:
# Create model (timm) and replace head for our number of classes
import torch.nn as nn
import torch.optim as optim
import timm

def create_vit_model(model_name='vit_base_patch16_224', num_classes=10, pretrained=True):
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    return model

# Example instantiation (user must set NUM_CLASSES)
# NUM_CLASSES = 10  # set appropriately from dataset classes
# model = create_vit_model(num_classes=NUM_CLASSES).to(device)
# print(model)


## Model Selection

We'll use a pretrained Vision Transformer from the `timm` library. Options:

1. **`vit_base_patch16_224`** (default, ~86M params) - Good baseline, widely used
2. **`deit_small_patch16_224`** (~22M params) - More efficient, trained with distillation
3. **`swin_small_patch4_window7_224`** (~50M params) - Hierarchical transformer, better for fine details
4. **`mobilevit_s`** (~5.6M params) - Lightweight, faster inference for real-time applications

For this notebook, we'll start with **ViT-B/16** and can easily swap to alternatives by changing `model_name`.

In [ ]:
# Training / validation loops (simple, clear)
import time
from sklearn.metrics import accuracy_score
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    preds = []
    targets = []
    for imgs, labels in dataloader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds.extend(outputs.argmax(dim=1).detach().cpu().numpy().tolist())
        targets.extend(labels.detach().cpu().numpy().tolist())
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(targets, preds)
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds = []
    targets = []
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().numpy().tolist())
            targets.extend(labels.cpu().numpy().tolist())
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(targets, preds)
    return epoch_loss, epoch_acc


## Training Configuration

Set hyperparameters for training.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import time

# Training hyperparameters
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
EPOCHS = 10
WARMUP_EPOCHS = 1

# Loss and optimizer
if os.path.exists(ANNOTATIONS_CSV):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    # Learning rate scheduler with warmup
    from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
    
    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
    main_scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, main_scheduler], milestones=[WARMUP_EPOCHS])
    
    print(f'✅ Training setup complete')
    print(f'   Learning rate: {LEARNING_RATE}')
    print(f'   Weight decay: {WEIGHT_DECAY}')
    print(f'   Epochs: {EPOCHS}')
    print(f'   Optimizer: AdamW with cosine annealing + warmup')

In [ ]:
# Small smoke-run orchestration (adjust paths and NUM_CLASSES first)
import math

,


## Training History

Plot loss and accuracy curves over epochs.

In [ ]:
if 'history' in locals() and history['train_loss']:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    # Loss
    axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train')
    axes[0].plot(epochs_range, history['val_loss'], 's-', label='Val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs_range, history['train_acc'], 'o-', label='Train')
    axes[1].plot(epochs_range, history['val_acc'], 's-', label='Val')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # F1 Score
    axes[2].plot(epochs_range, history['train_f1'], 'o-', label='Train')
    axes[2].plot(epochs_range, history['val_f1'], 's-', label='Val')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('F1 Score')
    axes[2].set_title('F1 Score (Macro)')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  No training history available. Train the model first.')

## Test Set Evaluation

Evaluate the best model on the held-out test set and generate a classification report and confusion matrix.

In [ ]:
import seaborn as sns

if os.path.exists(ANNOTATIONS_CSV) and os.path.exists('../checkpoints/vit_hagrid_best.pt'):
    # Load best checkpoint
    checkpoint = torch.load('../checkpoints/vit_hagrid_best.pt', map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'✅ Loaded best checkpoint (epoch {checkpoint["epoch"]}, val_acc={checkpoint["val_acc"]:.4f})')
    
    # Evaluate on test set
    test_loss, test_acc, test_f1, test_preds, test_targets = validate(
        model, test_loader, criterion, device
    )
    
    print(f'\n📊 Test Set Results:')
    print(f'   Loss: {test_loss:.4f}')
    print(f'   Accuracy: {test_acc:.4f}')
    print(f'   F1 Score (Macro): {test_f1:.4f}')
    
    # Classification report
    print(f'\n📋 Classification Report:')
    print(classification_report(test_targets, test_preds, target_names=train_dataset.classes, digits=4))
    
    # Confusion matrix
    cm = confusion_matrix(test_targets, test_preds)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=train_dataset.classes, 
                yticklabels=train_dataset.classes)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix (Test Set, Acc={test_acc:.4f})')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Checkpoint not found. Train the model first.')

## Next steps & notes
- Set `DATASET_PATH` to your HaGrid hand-crop dataset (recommended folder-per-class layout) and run the smoke test cell to validate pipeline.
- Consider alternatives to `vit_base_patch16_224` if you need lower latency: `swin_small_patch4_window7_224`, `deit_small_patch16_224`, or mobile-friendly backends like `mobilenetv3` or `mobilevit` variants (timm names vary).
- For larger experiments, switch to `torch.utils.tensorboard`, `wandb`, or HuggingFace `Trainer` for easier checkpointing and distributed training.
- After getting good accuracy, export model via `torch.jit.trace` or `onnx` and measure inference latency on your target device; consider quantization if needed.